In [1]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv("../data/raw/dataset.csv", encoding="latin1")
print(df.shape)
df.head()

(96011, 15)


C:\Users\DELL\AppData\Local\Temp\ipykernel_14928\78030392.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/dataset.csv", encoding="latin1")


,url,type,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,phishing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,phishing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,phishing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,mail.printakid.com/www.online.americanexpress....,phishing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,thewhiskeydregs.com/wp-content/themes/widescre...,phishing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
print(df.columns)


Index(['url', 'type', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10',
       'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14'],
      dtype='object')


In [11]:
df = df[['url', 'type']].copy()
df = df.rename(columns={'type': 'label'})

print(df.shape)
df.head()

(96011, 2)


,url,label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,phishing
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,phishing
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,phishing
3,mail.printakid.com/www.online.americanexpress....,phishing
4,thewhiskeydregs.com/wp-content/themes/widescre...,phishing


In [12]:
df['url'] = df['url'].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip().str.lower()

print(df['label'].unique())
print(df['label'].value_counts())

['phishing' 'nan' "0.7'8049" '0.770083' 'benign']
label
benign      48009
phishing    47904
nan            96
0.7'8049        1
0.770083        1
Name: count, dtype: int64


In [13]:
df['label'] = df['label'].map({
    'benign': 0,
    'phishing': 1
})

df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(df['label'].value_counts())

label
0    48009
1    47904
Name: count, dtype: int64


In [14]:
df = df[df['url'].notna()]
df = df[df['url'] != '']
df = df[df['url'] != 'nan']

print(df.shape)

(95913, 2)


In [15]:
def add_protocol(url):
    url = str(url).strip()
    if not url.startswith("http://") and not url.startswith("https://"):
        return "http://" + url
    return url

df['url'] = df['url'].apply(add_protocol)
df.head()

,url,label
0,http://nobell.it/70ffb52d079109dca5664cce6f317...,1
1,http://www.dghjdgf.com/paypal.co.uk/cycgi-bin/...,1
2,http://serviciosbys.com/paypal.cgi.bin.get-int...,1
3,http://mail.printakid.com/www.online.americane...,1
4,http://thewhiskeydregs.com/wp-content/themes/w...,1


In [16]:
before = len(df)
df = df.drop_duplicates(subset='url').reset_index(drop=True)
after = len(df)

print("Removed duplicates:", before - after)
print("New shape:", df.shape)

Removed duplicates: 2
New shape: (95911, 2)


In [17]:
print(df['label'].value_counts())

label
0    48009
1    47902
Name: count, dtype: int64


In [18]:
benign_df = df[df['label'] == 0]
phishing_df = df[df['label'] == 1]

print("Benign:", len(benign_df))
print("Phishing:", len(phishing_df))

Benign: 48009
Phishing: 47902


In [19]:
benign_sample = benign_df.sample(n=len(phishing_df), random_state=42)

df_balanced = pd.concat([benign_sample, phishing_df], axis=0)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced['label'].value_counts())
print(df_balanced.shape)

label
0    47902
1    47902
Name: count, dtype: int64
(95804, 2)


In [21]:
df_balanced.to_csv("../data/processed/dataset_balanced.csv", index=False)
print("Dataset saved successfully.")

Dataset saved successfully.


In [22]:
print(df['label'].unique())
print(df['label'].value_counts())

[1 0]
label
0    48009
1    47902
Name: count, dtype: int64
